# CIC-IDS-2017 feature preparation

This notebook starts from the cleaned dataset produced by `01_data_exploration.ipynb`. Its first stage removes identifier columns that would encourage memorization of the CIC-IDS-2017 laboratory, separates the remaining features from the multiclass target, and creates a reproducible stratified train/test split. No scaling, encoding, correlation filtering, resampling or learned feature selection is performed before the split.

## 1. Imports and paths

In [1]:
from itertools import combinations
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

processed_dataset_candidates = [
    Path("../data/processed/cicids2017_cleaned.parquet"),
    Path("ml/data/processed/cicids2017_cleaned.parquet"),
]

PROCESSED_DATA_PATH = next(
    (
        path.resolve()
        for path in processed_dataset_candidates
        if path.exists()
    ),
    None,
)

if PROCESSED_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cicids2017_cleaned.parquet. "
        "Run 01_data_exploration.ipynb first."
    )

RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f"Processed dataset: {PROCESSED_DATA_PATH}")
print(f"File size: {PROCESSED_DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MiB")
print(f"Random state: {RANDOM_STATE}")
print(f"Test fraction: {TEST_SIZE:.0%}")

Processed dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\processed\cicids2017_cleaned.parquet
File size: 360.48 MiB
Random state: 42
Test fraction: 20%


## 2. Load and validate the cleaned data

Confirm that the processed file matches the final dimensions and target structure recorded in the exploration notebook.

In [2]:
data = pd.read_parquet(PROCESSED_DATA_PATH)

EXPECTED_ROWS = 2_824_752
EXPECTED_COLUMNS = 76
EXPECTED_LABELS = 15

dataset_validation = pd.DataFrame(
    {
        "Observed": [
            data.shape[0],
            data.shape[1],
            data["Label"].nunique(dropna=False)
            if "Label" in data.columns
            else 0,
            int(data["Label"].isna().sum())
            if "Label" in data.columns
            else data.shape[0],
        ],
        "Expected": [
            EXPECTED_ROWS,
            EXPECTED_COLUMNS,
            EXPECTED_LABELS,
            0,
        ],
    },
    index=[
        "Rows",
        "Columns",
        "Detailed target classes",
        "Missing target labels",
    ],
)
display(dataset_validation)

if not dataset_validation["Observed"].equals(
    dataset_validation["Expected"]
):
    raise AssertionError(
        "The cleaned dataset does not match the expected EDA output."
    )

print("The cleaned dataset matches the final EDA output.")

,Observed,Expected
Rows,2824752,2824752
Columns,76,76
Detailed target classes,15,15
Missing target labels,0,0


The cleaned dataset matches the final EDA output.


## 3. Review all cleaned columns

Display the complete cleaned schema before excluding identifiers or separating the target.

In [3]:
cleaned_column_overview = pd.DataFrame(
    {
        "Position": range(1, len(data.columns) + 1),
        "Column": data.columns,
        "Data type": data.dtypes.astype(str).to_numpy(),
    }
).set_index("Position")

with pd.option_context("display.max_rows", None):
    display(cleaned_column_overview)
print(f"Total cleaned columns: {len(cleaned_column_overview)}")

,Column,Data type
Position,,
1,Flow ID,object
2,Src IP,object
3,Src Port,float64
4,Dst IP,object
5,Dst Port,float64
6,Protocol,float64
7,Timestamp,object
8,Flow Duration,float64
9,Total Fwd Packet,float64


Total cleaned columns: 76


## 4. Remove identifier and leakage-prone columns

`Flow ID`, endpoint IP addresses and `Timestamp` describe this particular laboratory capture rather than transferable flow behavior. These fixed schema exclusions do not depend on statistics calculated from the dataset, so remove them directly from the cleaned table before separating features and target. Ports and `Protocol` are retained for now.

In [4]:
identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp",
]
missing_identifier_columns = [
    column for column in identifier_columns if column not in data.columns
]
if missing_identifier_columns:
    raise KeyError(
        "Expected identifier columns are missing: "
        f"{missing_identifier_columns}"
    )

identifier_decisions = pd.DataFrame(
    {
        "Column": identifier_columns,
        "Reason for exclusion": [
            "Constructed flow identifier; encourages record memorization",
            "Source identity is specific to the CIC-IDS-2017 laboratory",
            "Destination identity is specific to the CIC-IDS-2017 laboratory",
            "Encodes the published attack schedule and capture day",
        ],
    }
)
display(identifier_decisions)

data.drop(columns=identifier_columns, inplace=True)
print(f"Columns remaining after fixed exclusions: {data.shape[1]}")

,Column,Reason for exclusion
0,Flow ID,Constructed flow identifier; encourages record...
1,Src IP,Source identity is specific to the CIC-IDS-201...
2,Dst IP,Destination identity is specific to the CIC-ID...
3,Timestamp,Encodes the published attack schedule and capt...


Columns remaining after fixed exclusions: 72


## 5. Separate features and target

Keep the original 15-class `Label` as the target. Using `pop` separates it without duplicating the large feature table in memory.

In [5]:
y = data.pop("Label")
X = data
del data

print(f"Feature rows: {X.shape[0]:,}")
print(f"Feature columns: {X.shape[1]}")
print(f"Target rows: {len(y):,}")
print(f"Target classes: {y.nunique()}")

Feature rows: 2,824,752
Feature columns: 71
Target rows: 2,824,752
Target classes: 15


## 6. Inspect Protocol values

Start by displaying the unmodified `Protocol.value_counts()` result. Assign readable names only after the values actually present in the cleaned dataset are visible. The stored `Protocol` column is not changed in this section.

In [6]:
raw_protocol_counts = (
    X["Protocol"].value_counts(dropna=False).sort_index()
)
raw_protocol_distribution = (
    raw_protocol_counts
    .rename_axis("Raw Protocol value")
    .rename("Flow count")
    .to_frame()
)
raw_protocol_distribution["Percentage of dataset"] = (
    raw_protocol_distribution["Flow count"] / len(X) * 100
)
display(raw_protocol_distribution)

,Flow count,Percentage of dataset
Raw Protocol value,,
0.0,1696,0.060041
6.0,1823631,64.558977
17.0,999425,35.380982


The observed protocol values and their names are:

- `0`: HOPOPT
- `6`: TCP
- `17`: UDP

The notebook leaves the `Protocol` column unchanged. `Label` is the target rather than a feature, and the representation of source and destination ports remains a later feature-engineering decision.

## 7. Create the train/test split

Use a reproducible stratified 80/20 split so every target class is represented in both subsets. This split estimates performance on held-out flows from the same CIC-IDS-2017 capture; it does not by itself demonstrate generalization to a different network or time period. All later data-dependent preprocessing must be fitted using `X_train` and `y_train` only.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

if not X_train.index.intersection(X_test.index).empty:
    raise AssertionError("Training and test rows overlap.")

del X, y

split_size_summary = pd.DataFrame(
    {
        "Rows": [len(X_train), len(X_test)],
        "Percentage of dataset": [
            len(X_train) / (len(X_train) + len(X_test)) * 100,
            len(X_test) / (len(X_train) + len(X_test)) * 100,
        ],
    },
    index=["Training set", "Test set"],
)
display(split_size_summary)
print(f"Features in each split: {X_train.shape[1]}")

,Rows,Percentage of dataset
Training set,2259801,79.999979
Test set,564951,20.000021


Features in each split: 71


## 8. Verify split distributions

Confirm that no class disappeared and quantify how closely stratification preserved the original multiclass distribution.

In [8]:
training_label_counts = y_train.value_counts()
test_label_counts = y_test.value_counts()
total_label_counts = (
    training_label_counts.add(test_label_counts, fill_value=0)
    .astype("int64")
    .sort_values(ascending=False)
)

split_label_distribution = pd.DataFrame(
    {
        "Total flows": total_label_counts,
        "Training flows": training_label_counts.reindex(
            total_label_counts.index, fill_value=0
        ),
        "Test flows": test_label_counts.reindex(
            total_label_counts.index, fill_value=0
        ),
    }
)
split_label_distribution["Training percentage"] = (
    split_label_distribution["Training flows"] / len(y_train) * 100
)
split_label_distribution["Test percentage"] = (
    split_label_distribution["Test flows"] / len(y_test) * 100
)
split_label_distribution["Absolute difference (percentage points)"] = (
    split_label_distribution["Training percentage"]
    .sub(split_label_distribution["Test percentage"])
    .abs()
)
display(split_label_distribution)

if split_label_distribution[["Training flows", "Test flows"]].eq(0).any().any():
    raise AssertionError("At least one target class is absent from a split.")

if not (
    split_label_distribution["Training flows"]
    + split_label_distribution["Test flows"]
).equals(split_label_distribution["Total flows"]):
    raise AssertionError("Split label counts do not reproduce the full target.")

print("All 15 classes are present in both splits.")
print("The train/test split is complete; no preprocessing has been fitted.")

,Total flows,Training flows,Test flows,Training percentage,Test percentage,Absolute difference (percentage points)
Label,,,,,,
BENIGN,2268391,1814712,453679,80.304062,80.304133,7.039894e-05
DoS Hulk,229964,183971,45993,8.141027,8.141060,3.344402e-05
PortScan,158804,127043,31761,5.621867,5.621903,3.678832e-05
DDoS,128006,102405,25601,4.531594,4.531543,5.026754e-05
DoS GoldenEye,10288,8230,2058,0.364191,0.364279,8.801977e-05
FTP-Patator,7931,6345,1586,0.280777,0.280732,4.462437e-05
SSH-Patator,5895,4716,1179,0.208691,0.208691,2.770474e-07
DoS slowloris,5796,4637,1159,0.205195,0.205151,4.452403e-05
DoS Slowhttptest,5499,4399,1100,0.194663,0.194707,4.399320e-05


All 15 classes are present in both splits.
The train/test split is complete; no preprocessing has been fitted.


## 9. Calculate feature-to-label mutual information

Calculate mutual information between every feature in `X_train` and the multiclass target `y_train`. `Protocol` is identified as discrete only for the estimator; its stored dtype and values remain unchanged. Display the complete ranking without selecting or removing features.

In [9]:
mi_feature_columns = X_train.columns.tolist()
mi_discrete_feature_mask = [
    column == "Protocol" for column in mi_feature_columns
]

mi_scores = mutual_info_classif(
    X_train[mi_feature_columns],
    y_train,
    discrete_features=mi_discrete_feature_mask,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

mutual_information_ranking = pd.DataFrame(
    {
        "Feature": mi_feature_columns,
        "Mutual information": mi_scores,
        "MI estimator input type": [
            "Discrete" if is_discrete else "Continuous"
            for is_discrete in mi_discrete_feature_mask
        ],
    }
).sort_values(
    "Mutual information", ascending=False, ignore_index=True
)
mutual_information_ranking.index = pd.RangeIndex(
    start=1,
    stop=len(mutual_information_ranking) + 1,
    name="Rank",
)

print(f"Features scored: {len(mutual_information_ranking)}")
with pd.option_context("display.max_rows", None):
    display(mutual_information_ranking)

Features scored: 71


,Feature,Mutual information,MI estimator input type
Rank,,,
1,Average Packet Size,0.589162,Continuous
2,Packet Length Mean,0.561130,Continuous
3,Packet Length Std,0.558339,Continuous
4,Packet Length Variance,0.556995,Continuous
5,Subflow Bwd Bytes,0.498750,Continuous
6,Total Length of Bwd Packet,0.498369,Continuous
7,FWD Init Win Bytes,0.497415,Continuous
8,Bwd Packet Length Mean,0.490913,Continuous
9,Bwd Segment Size Avg,0.490572,Continuous


The highest MI scores belong mainly to overall packet-length statistics, transferred-byte features, TCP initial-window values and timing/rate features. `Average Packet Size` ranks first (`0.589`), followed by `Packet Length Mean`, `Packet Length Std` and `Packet Length Variance`. Several known related features receive almost equal scores, such as total and subflow byte values and directional packet-length means and segment-size averages. `Protocol` ranks 61st (`0.099`). The lowest scores belong mostly to sparse flag counters, with `CWR Flag Count`, `RST Flag Count` and `ECE Flag Count` estimated at zero. These scores provide a ranking, not a removal threshold; global MI can undervalue signals associated only with the rarest classes.

## 10. Calculate Pearson feature correlations

Calculate Pearson correlations using `X_train` only. Exclude `Protocol` because its numeric values are protocol identifiers rather than continuous measurements. Calculate every unique pair, but display only the top 100 in descending order of absolute correlation. The table is an inspection aid: correlation alone does not authorize removing either feature.

In [10]:
TOP_CORRELATION_PAIRS_TO_DISPLAY = 100
correlation_feature_columns = [
    column for column in X_train.columns if column != "Protocol"
]

pearson_correlation_matrix = X_train[
    correlation_feature_columns
].corr(method="pearson")

pearson_correlation_pairs = pd.DataFrame(
    (
        (
            first_feature,
            second_feature,
            pearson_correlation_matrix.at[
                first_feature, second_feature
            ],
        )
        for first_feature, second_feature in combinations(
            correlation_feature_columns, 2
        )
    ),
    columns=["Feature 1", "Feature 2", "Pearson correlation"],
)
pearson_correlation_pairs["Absolute correlation"] = (
    pearson_correlation_pairs["Pearson correlation"].abs()
)
pearson_correlation_pairs.sort_values(
    "Absolute correlation", ascending=False, inplace=True
)
pearson_correlation_pairs.index = pd.RangeIndex(
    start=1,
    stop=len(pearson_correlation_pairs) + 1,
    name="Pair",
)

print(f"Training features included: {len(correlation_feature_columns)}")
print(f"Unique feature pairs checked: {len(pearson_correlation_pairs):,}")
print(f"Top {TOP_CORRELATION_PAIRS_TO_DISPLAY} pairs are displayed below.")

pearson_correlation_display = pearson_correlation_pairs.head(
    TOP_CORRELATION_PAIRS_TO_DISPLAY
).copy()
pearson_correlation_display["Pearson correlation"] = (
    pearson_correlation_display["Pearson correlation"].round(6)
)
pearson_correlation_display["Absolute correlation"] = (
    pearson_correlation_display["Absolute correlation"].round(6)
)
with pd.option_context("display.max_rows", None):
    display(pearson_correlation_display)

Training features included: 70
Unique feature pairs checked: 2,415
Top 100 pairs are displayed below.


,Feature 1,Feature 2,Pearson correlation,Absolute correlation
Pair,,,,
1,Bwd Packet Length Mean,Bwd Segment Size Avg,1.000000,1.000000
2,Fwd URG Flags,CWR Flag Count,1.000000,1.000000
3,Total Fwd Packet,Subflow Fwd Packets,1.000000,1.000000
4,Fwd Packet Length Mean,Fwd Segment Size Avg,1.000000,1.000000
5,Fwd PSH Flags,SYN Flag Count,1.000000,1.000000
6,Total Bwd packets,Subflow Bwd Packets,1.000000,1.000000
7,Total Length of Bwd Packet,Subflow Bwd Bytes,1.000000,1.000000
8,Total Length of Fwd Packet,Subflow Fwd Bytes,0.999999,0.999999
9,Flow Duration,Fwd IAT Total,0.998572,0.998572


Pearson was calculated for all 2,415 unique pairs among the 70 included training features, while the output shows the 100 strongest pairs. The strongest relationships mainly form recurring clusters:

- forward/backward packet and byte totals, subflow values, header lengths and active-data counts;
- packet-length means, segment-size averages and other packet-length summaries;
- flow duration, inter-arrival-time maxima and idle-period statistics.

Several differently defined flag counters are also almost or exactly correlated in this dataset, including `Fwd PSH Flags` with `SYN Flag Count`, `Fwd URG Flags` with `CWR Flag Count`, and `RST Flag Count` with `ECE Flag Count`. This is evidence about their behavior in CIC-IDS-2017, not evidence that the features have the same meaning. Therefore no feature is removed from this table alone; the correlated clusters must be combined with feature meaning and target relevance before choosing what to retain.

## 11. Calculate Spearman feature correlations

Calculate Spearman rank correlations using the same 70 training features as Pearson. Spearman detects strong monotonic relationships even when their shape is nonlinear. Calculate every unique pair, but display only the top 100 without selecting or removing features.

In [11]:
spearman_correlation_matrix = X_train[
    correlation_feature_columns
].corr(method="spearman")

spearman_correlation_pairs = pd.DataFrame(
    (
        (
            first_feature,
            second_feature,
            spearman_correlation_matrix.at[
                first_feature, second_feature
            ],
        )
        for first_feature, second_feature in combinations(
            correlation_feature_columns, 2
        )
    ),
    columns=["Feature 1", "Feature 2", "Spearman correlation"],
)
spearman_correlation_pairs["Absolute correlation"] = (
    spearman_correlation_pairs["Spearman correlation"].abs()
)
spearman_correlation_pairs.sort_values(
    "Absolute correlation", ascending=False, inplace=True
)
spearman_correlation_pairs.index = pd.RangeIndex(
    start=1,
    stop=len(spearman_correlation_pairs) + 1,
    name="Pair",
)

print(f"Training features included: {len(correlation_feature_columns)}")
print(f"Unique feature pairs checked: {len(spearman_correlation_pairs):,}")
print(f"Top {TOP_CORRELATION_PAIRS_TO_DISPLAY} pairs are displayed below.")

spearman_correlation_display = spearman_correlation_pairs.head(
    TOP_CORRELATION_PAIRS_TO_DISPLAY
).copy()
spearman_correlation_display["Spearman correlation"] = (
    spearman_correlation_display["Spearman correlation"].round(6)
)
spearman_correlation_display["Absolute correlation"] = (
    spearman_correlation_display["Absolute correlation"].round(6)
)
with pd.option_context("display.max_rows", None):
    display(spearman_correlation_display)

Training features included: 70
Unique feature pairs checked: 2,415
Top 100 pairs are displayed below.


,Feature 1,Feature 2,Spearman correlation,Absolute correlation
Pair,,,,
1,Total Length of Bwd Packet,Subflow Bwd Bytes,1.000000,1.000000
2,Fwd URG Flags,CWR Flag Count,1.000000,1.000000
3,Total Fwd Packet,Subflow Fwd Packets,1.000000,1.000000
4,Total Bwd packets,Subflow Bwd Packets,1.000000,1.000000
5,Total Length of Fwd Packet,Subflow Fwd Bytes,1.000000,1.000000
6,Fwd PSH Flags,SYN Flag Count,1.000000,1.000000
7,Bwd Packet Length Mean,Bwd Segment Size Avg,1.000000,1.000000
8,Fwd Packet Length Mean,Fwd Segment Size Avg,1.000000,1.000000
9,Packet Length Std,Packet Length Variance,0.999984,0.999984


Spearman exposes strong monotonic relationships that Pearson does not capture well. Notable examples include `Packet Length Std` with `Packet Length Variance` (Pearson `0.925`, Spearman `1.000`), `Active Mean` with `Active Max` (Pearson `0.830`, Spearman `0.999`), and `Flow Packets/s` with `Flow IAT Mean` (Pearson `-0.080`, Spearman `-0.997`). The exact total/subflow and directional mean/segment-size relationships remain at or near one under both methods. This demonstrates that Pearson alone is insufficient for the redundancy audit, but a high Spearman value still identifies a candidate relationship rather than authorizing deletion.

## 12. Combine MI, Pearson and Spearman results

Join the two complete correlation results with the feature-to-label MI scores. Sort all pairs by the larger of their absolute Pearson and Spearman correlations, but display only the top 100. No filtering rule is applied.

In [12]:
mi_score_by_feature = mutual_information_ranking.set_index(
    "Feature"
)["Mutual information"]

combined_filter_results = (
    pearson_correlation_pairs[
        ["Feature 1", "Feature 2", "Pearson correlation"]
    ]
    .merge(
        spearman_correlation_pairs[
            ["Feature 1", "Feature 2", "Spearman correlation"]
        ],
        on=["Feature 1", "Feature 2"],
        how="inner",
        validate="one_to_one",
    )
)
combined_filter_results.insert(
    1,
    "Feature 1 MI",
    combined_filter_results["Feature 1"].map(mi_score_by_feature),
)
combined_filter_results.insert(
    3,
    "Feature 2 MI",
    combined_filter_results["Feature 2"].map(mi_score_by_feature),
)
combined_filter_results["Absolute Pearson correlation"] = (
    combined_filter_results["Pearson correlation"].abs()
)
combined_filter_results["Absolute Spearman correlation"] = (
    combined_filter_results["Spearman correlation"].abs()
)
combined_filter_results["Maximum absolute correlation"] = (
    combined_filter_results[
        [
            "Absolute Pearson correlation",
            "Absolute Spearman correlation",
        ]
    ].max(axis=1)
)
combined_filter_results.sort_values(
    "Maximum absolute correlation",
    ascending=False,
    inplace=True,
)
combined_filter_results.index = pd.RangeIndex(
    start=1,
    stop=len(combined_filter_results) + 1,
    name="Pair",
)

print(f"Combined feature pairs: {len(combined_filter_results):,}")
print(f"Top {TOP_CORRELATION_PAIRS_TO_DISPLAY} pairs are displayed below.")

combined_filter_display = combined_filter_results.head(
    TOP_CORRELATION_PAIRS_TO_DISPLAY
).copy()
numeric_result_columns = [
    column
    for column in combined_filter_display.columns
    if column not in ["Feature 1", "Feature 2"]
]
combined_filter_display[numeric_result_columns] = (
    combined_filter_display[numeric_result_columns].round(6)
)
with pd.option_context("display.max_rows", None):
    display(combined_filter_display)

Combined feature pairs: 2,415
Top 100 pairs are displayed below.


,Feature 1,Feature 1 MI,Feature 2,Feature 2 MI,Pearson correlation,Spearman correlation,Absolute Pearson correlation,Absolute Spearman correlation,Maximum absolute correlation
Pair,,,,,,,,,
1,Bwd Packet Length Mean,0.490913,Bwd Segment Size Avg,0.490572,1.000000,1.000000,1.000000,1.000000,1.000000
2,Total Length of Bwd Packet,0.498369,Subflow Bwd Bytes,0.498750,1.000000,1.000000,1.000000,1.000000,1.000000
3,Total Length of Fwd Packet,0.485725,Subflow Fwd Bytes,0.485743,0.999999,1.000000,0.999999,1.000000,1.000000
4,Fwd Packet Length Mean,0.401005,Fwd Segment Size Avg,0.401476,1.000000,1.000000,1.000000,1.000000,1.000000
5,Total Bwd packets,0.270560,Subflow Bwd Packets,0.270831,1.000000,1.000000,1.000000,1.000000,1.000000
6,Fwd URG Flags,0.000021,CWR Flag Count,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,Fwd PSH Flags,0.019300,SYN Flag Count,0.018975,1.000000,1.000000,1.000000,1.000000,1.000000
8,Total Fwd Packet,0.247153,Subflow Fwd Packets,0.246821,1.000000,1.000000,1.000000,1.000000,1.000000
9,Packet Length Std,0.558339,Packet Length Variance,0.556995,0.924830,0.999984,0.924830,0.999984,0.999984


The combined calculation contains all 2,415 pairs, while the output shows the 100 strongest according to the larger absolute Pearson or Spearman value. It shows where a future choice may be straightforward—for example, the standard-deviation and variance features are almost perfectly rank-correlated and have nearly equal MI—and where MI alone cannot decide, such as flag pairs whose scores are both approximately zero. No columns are removed here. The next stage must define candidate rules and compare the resulting feature sets through training-only model validation.

## 13. Propose correlation-based feature removals

Create a candidate feature set without modifying either split. A pair is considered highly associated when its absolute Pearson or Spearman correlation is at least `0.99`. Process features from highest to lowest MI: retain a feature unless it is highly associated with a feature already retained. When a conflict exists, record the higher-MI retained feature as the proposed alternative. This greedy procedure handles overlapping pairs without independently deleting one feature for every table row.

In [13]:
HIGH_CORRELATION_THRESHOLD = 0.99

high_correlation_relationships = combined_filter_results.loc[
    (
        combined_filter_results["Absolute Pearson correlation"]
        >= HIGH_CORRELATION_THRESHOLD
    )
    | (
        combined_filter_results["Absolute Spearman correlation"]
        >= HIGH_CORRELATION_THRESHOLD
    )
].copy()

correlation_connections = {}
for _, relationship in high_correlation_relationships.iterrows():
    first_feature = relationship["Feature 1"]
    second_feature = relationship["Feature 2"]
    correlation_connections.setdefault(first_feature, []).append(
        (second_feature, relationship)
    )
    correlation_connections.setdefault(second_feature, []).append(
        (first_feature, relationship)
    )

mi_score_by_feature = mutual_information_ranking.set_index(
    "Feature"
)["Mutual information"]
mi_ranked_features = mutual_information_ranking["Feature"].tolist()
proposed_retained_features = []
proposed_retained_feature_set = set()
removal_candidate_records = []

for feature in mi_ranked_features:
    retained_conflicts = [
        (other_feature, relationship)
        for other_feature, relationship
        in correlation_connections.get(feature, [])
        if other_feature in proposed_retained_feature_set
    ]

    if not retained_conflicts:
        proposed_retained_features.append(feature)
        proposed_retained_feature_set.add(feature)
        continue

    retained_feature, strongest_relationship = max(
        retained_conflicts,
        key=lambda item: item[1]["Maximum absolute correlation"],
    )
    removal_candidate_records.append(
        {
            "Removal candidate": feature,
            "Candidate MI": mi_score_by_feature[feature],
            "Proposed retained feature": retained_feature,
            "Retained MI": mi_score_by_feature[retained_feature],
            "MI advantage of retained feature": (
                mi_score_by_feature[retained_feature]
                - mi_score_by_feature[feature]
            ),
            "Pearson correlation": strongest_relationship[
                "Pearson correlation"
            ],
            "Spearman correlation": strongest_relationship[
                "Spearman correlation"
            ],
            "Maximum absolute correlation": strongest_relationship[
                "Maximum absolute correlation"
            ],
            "Correlated retained feature count": len(
                retained_conflicts
            ),
        }
    )

proposed_removal_candidates = pd.DataFrame(removal_candidate_records)
proposed_removal_candidates.sort_values(
    ["Maximum absolute correlation", "Candidate MI"],
    ascending=[False, False],
    inplace=True,
)
proposed_removal_candidates.index = pd.RangeIndex(
    start=1,
    stop=len(proposed_removal_candidates) + 1,
    name="Candidate",
)

proposed_retained_feature_table = mutual_information_ranking.loc[
    mutual_information_ranking["Feature"].isin(
        proposed_retained_feature_set
    )
].copy()

print(f"High-correlation relationships: {len(high_correlation_relationships)}")
print(f"Original features: {X_train.shape[1]}")
print(f"Proposed retained features: {len(proposed_retained_features)}")
print(f"Proposed removal candidates: {len(proposed_removal_candidates)}")
print("X_train and X_test remain unchanged.")

print("Proposed removal candidates:")
with pd.option_context("display.max_rows", None):
    display(proposed_removal_candidates.round(6))

print("Proposed retained features, ordered by MI rank:")
with pd.option_context("display.max_rows", None):
    display(proposed_retained_feature_table.round(6))

High-correlation relationships: 39
Original features: 71
Proposed retained features: 48
Proposed removal candidates: 23
X_train and X_test remain unchanged.
Proposed removal candidates:


,Removal candidate,Candidate MI,Proposed retained feature,Retained MI,MI advantage of retained feature,Pearson correlation,Spearman correlation,Maximum absolute correlation,Correlated retained feature count
Candidate,,,,,,,,,
1,Bwd Segment Size Avg,0.490572,Bwd Packet Length Mean,0.490913,0.000341,1.000000,1.000000,1.000000,1
2,Total Length of Bwd Packet,0.498369,Subflow Bwd Bytes,0.498750,0.000380,1.000000,1.000000,1.000000,1
3,Total Length of Fwd Packet,0.485725,Subflow Fwd Bytes,0.485743,0.000018,0.999999,1.000000,1.000000,1
4,Fwd Packet Length Mean,0.401005,Fwd Segment Size Avg,0.401476,0.000470,1.000000,1.000000,1.000000,1
5,Subflow Fwd Packets,0.246821,Total Fwd Packet,0.247153,0.000332,1.000000,1.000000,1.000000,1
6,SYN Flag Count,0.018975,Fwd PSH Flags,0.019300,0.000326,1.000000,1.000000,1.000000,1
7,CWR Flag Count,0.000000,Fwd URG Flags,0.000021,0.000021,1.000000,1.000000,1.000000,1
8,Packet Length Variance,0.556995,Packet Length Std,0.558339,0.001344,0.924830,0.999984,0.999984,1
9,Idle Mean,0.199806,Idle Max,0.203883,0.004077,0.990370,0.999575,0.999575,1


Proposed retained features, ordered by MI rank:


,Feature,Mutual information,MI estimator input type
Rank,,,
1,Average Packet Size,0.589162,Continuous
3,Packet Length Std,0.558339,Continuous
5,Subflow Bwd Bytes,0.498750,Continuous
7,FWD Init Win Bytes,0.497415,Continuous
8,Bwd Packet Length Mean,0.490913,Continuous
10,Subflow Fwd Bytes,0.485743,Continuous
12,Bwd Packet Length Max,0.457174,Continuous
13,Packet Length Max,0.454000,Continuous
14,Bwd Init Win Bytes,0.448389,Continuous


The `0.99` rule finds 39 high-correlation relationships. The MI-ordered greedy procedure proposes retaining 48 of the 71 features and marks 23 as removal candidates, while leaving both data splits unchanged. Some proposals describe direct or near-direct redundancy, including directional packet-length means versus segment-size averages, total versus subflow measurements, and packet-length standard deviation versus variance.

The table is not yet a safe deletion list. Several exact pairs have negligible MI differences, so estimator noise or feature clarity may be more important than which score is microscopically larger. The flag pairs have different meanings despite matching behavior in this dataset. Other proposals are driven mainly by Spearman—for example, `Flow IAT Mean` versus `Fwd Packets/s` and `Flow Duration` versus `Flow IAT Max`—and identify monotonic association rather than interchangeable measurements. Likewise, packet-count features should not be discarded solely because they correlate with header length. These cases require review before creating the reduced feature set, and the eventual proposal must be compared with all 71 features through training-only validation.